# Notebook 4 – Segmentation par Deep Learning (U-Net)

**PFE 2025/2026** – Segmentation d'images en présence de bruit  

---

Ce notebook présente :
1. L'architecture U-Net
2. L'entraînement avec augmentation de bruit
3. L'inférence sur images bruitées
4. Comparaison U-Net vs méthodes classiques

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from segmentation.deep_learning import UNet

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 4.1 Architecture U-Net

In [ ]:
model = UNet(in_channels=3, out_channels=1)

# Compter les paramètres
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Paramètres totaux    : {total_params:,}')
print(f'Paramètres entraîn. : {trainable:,}')
print()
print(model)

## 4.2 Test forward pass

In [ ]:
model.eval()
dummy_input = torch.randn(2, 3, 256, 256)  # batch=2, RGB, 256×256
with torch.no_grad():
    output = model(dummy_input)

print(f'Input shape  : {dummy_input.shape}')
print(f'Output shape : {output.shape}')   # doit être [2, 1, 256, 256]
print(f'Output range : [{output.min():.3f}, {output.max():.3f}]')

## 4.3 Entraînement

> **Pré-requis** : Placer des images dans `data/images/` et les masques correspondants dans `data/masks/` (même nom de fichier).

Décommenter la cellule suivante pour lancer l'entraînement.

In [ ]:
# from segmentation.deep_learning import train_unet
# import config
#
# history = train_unet(
#     image_dir       = config.IMAGE_DIR,
#     mask_dir        = config.MASK_DIR,
#     checkpoint_path = config.UNET_CHECKPOINT,
#     image_size      = 256,
#     batch_size      = 4,
#     epochs          = 30,
#     lr              = 1e-4,
#     noise_type      = 'gaussian',
#     noise_params    = {'sigma': 25},
# )
print('Entraînement désactivé – décommentez le code ci-dessus pour lancer.')

## 4.4 Visualisation des prédictions U-Net

In [ ]:
import cv2
from pathlib import Path
import config

checkpoint = config.UNET_CHECKPOINT

if checkpoint.exists():
    from segmentation.deep_learning.predict import load_model, predict
    from noise import add_gaussian_noise, add_salt_and_pepper_noise
    
    loaded_model = load_model(checkpoint, device=device)
    
    data_dir = Path('../data/images')
    imgs = list(data_dir.glob('*.png')) + list(data_dir.glob('*.jpg'))
    
    if imgs:
        image = cv2.imread(str(imgs[0]))
        
        noisy_g  = add_gaussian_noise(image, sigma=40)
        noisy_sp = add_salt_and_pepper_noise(image, density=0.10)
        
        pred_clean = predict(loaded_model, image,    device=device)
        pred_g     = predict(loaded_model, noisy_g,  device=device)
        pred_sp    = predict(loaded_model, noisy_sp, device=device)
        
        fig, axes = plt.subplots(2, 3, figsize=(12, 7))
        for ax, img, title in zip(axes[0], [image, noisy_g, noisy_sp],
                                   ['Original', 'Gaussien σ=40', 'S&P d=0.10']):
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(title)
            ax.axis('off')
        for ax, mask, title in zip(axes[1], [pred_clean, pred_g, pred_sp],
                                    ['Pred. clean', 'Pred. Gaussien', 'Pred. S&P']):
            ax.imshow(mask, cmap='gray')
            ax.set_title(title)
            ax.axis('off')
        plt.suptitle('Prédictions U-Net', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print('Aucune image dans data/images/')
else:
    print(f'Checkpoint introuvable : {checkpoint}  →  Lancez d\'abord l\'entraînement.')

## 4.5 Fonction de perte : BCE + Dice

In [ ]:
# Visualisation de la surface de perte Dice
import numpy as np

prec_vals = np.linspace(0.01, 1, 50)
rec_vals  = np.linspace(0.01, 1, 50)
P, R = np.meshgrid(prec_vals, rec_vals)
F1 = 2 * P * R / (P + R)

fig = plt.figure(figsize=(7, 5))
ax  = fig.add_subplot(111)
cp  = ax.contourf(P, R, F1, levels=20, cmap='RdYlGn')
plt.colorbar(cp, label='F1 / Dice')
ax.set_xlabel('Précision')
ax.set_ylabel('Rappel')
ax.set_title('Score F1 = Dice en fonction de la Précision et du Rappel')
plt.tight_layout()
plt.savefig('../results/figures/f1_surface.png', dpi=150, bbox_inches='tight')
plt.show()